Help : https://cengel.github.io/R-spatial/spatialops.html

In [ ]:
system("conda install -y conda-forge::r-rcpp conda-forge::openssl conda-forge::r-sf conda-forge::r-terra conda-forge::r-ncdf4")
system("conda install -y conda-forge::r-r.utils conda-forge::r-tidyverse conda-forge::libgdal-hdf5 conda-forge::r-ggplot2")
system("conda install -y conda-forge::r-lubridate conda-forge::r-rcolorbrewer conda-forge::r-lattice conda-forge::r-png r::r-raster conda-forge::r-fnn")
system("conda install -y conda-forge::r-cluster conda-forge::r-remotes conda-forge::r-devtools")
system("conda install -y conda-forge::r-factominer conda-forge::r-caret conda-forge::r-factoextra conda-forge::r-rlang")
system("conda install -y conda-forge::r-geojsonio")


In [ ]:
library(ncdf4)
library(R.utils)
library(tidyverse) # because who can live without the tidyverse?
library(terra)     
library(dplyr)
        
library(jsonlite) 
library(utils)
library(ggplot2)

library(ncdf4) #     ncdf4: open, write and create NetCDF files (also provides metadata information)
library(lubridate) # lubridate: operate on date and times data
library(RColorBrewer) # RColorBrewer: create colour palettes for thematic maps
library(lattice) # lattice : visualization system for typical graphics

library(data.table)
library(FNN)

library(cluster)
# library(bioregion)

# library(rlang)
# library(factoextra)

library(sf)
library(geojsonio)

In [ ]:
# SET DIRECTORIES
workdir <- getwd()
dataDir <- paste(workdir,"Data",sep = "/")
outputDir <- paste(workdir,"outputs/collection",sep = "/")
scriptDir <- paste(workdir,"scripts",sep = "/")

In [ ]:
# GENERAL FUNCTIONS

# ==============================================================================
# CREATE REPERTORY
# ==============================================================================
create.directory <- function(name.directory){
    ifelse(!dir.exists(file.path(name.directory)),
        dir.create(file.path(name.directory)),
        "Directory Exists")
}

# ==============================================================================
# DOWNLOAD FILENAME
# ==============================================================================
download.filename <- function(filename, url){
    options(timeout = 600)  # 10 minutes
    
    if(file.exists(filename)){
        cat(filename, "is (are) already in your repertory.")
    } else {
        download.file(url, filename, mode = "wb")
        print('File Downloaded')
    }
}

# ==============================================================================
# UNZIP FILENAME
# ==============================================================================
unzip.file <- function(filename, type = "gz"){
    filename.length <- nchar(filename)
    print(filename, filename.length)

    start <- 1

    if(type == "gz"){
        # Case Gunzip
        end <- filename.length-3
    }
    else{
        # Case Unzip
        end <- filename.length-4
    }

    unzip.filename <- substr(filename,start, end)
    print(unzip.filename)

    
    # Case gunzip
    if(!file.exists(unzip.filename)){
        if(type == "gz"){
            R.utils::gunzip(filename, overwrite=FALSE, remove=TRUE, BFR.SIZE=1e+07)
            }
        else{
            utils::unzip(filename, overwrite=FALSE )
            }
        cat(filename, "successfully unzipped!")
    }

    return(unzip.filename)
}

# ==============================================================================
# EXPORT AS A CSV 
# ==============================================================================
export.csv <- function(directory = outputDir, table.output, filename){
    table_filename <- paste(directory,filename, sep="/")
    write.table(table.output, table_filename, row.names=TRUE, col.names = TRUE, sep=",")
    print(paste(filename,"output exported.", sep = " "))
    }
    
# ==============================================================================
# MOVE TO DATA DIRECTORY 
# ==============================================================================
move.file <- function(filename, new.path){
    new.filename <- paste(new.path, filename, sep = "/")
    file.rename(from=filename, to=new.filename)
    return(new.filename)
}


In [ ]:
gdb_path_zip <- "Geomorphology.gdb.zip"

download.filename(url = "https://d28rz98at9flks.cloudfront.net/102441/Geomorphology.gdb.zip",
                 filename = gdb_path_zip) # https://data.gov.au/data/dataset/geomorphic-features-of-the-antarctic-margin-and-southern-ocean-20126/resource/cedf76d9-4390-4080-b9d0-84c57bd1c7b8

gdb_path <- unzip.file(gdb_path_zip, type = "zip")

# gdb_path <- move.file(gdb_path, paste(dataDir,"Geomorphology.gdb",sep = "/"))


In [ ]:
#Correct unzipped path
gdb_path <- unlist(strsplit(gdb_path, "/"))[[length( unlist(strsplit(gdb_path, "/")) )]]
print(gdb_path)

In [ ]:
# https://gis.stackexchange.com/questions/426282/load-gdb-directory-into-r-using-simple-features-package
model <- sf::st_read(dsn = gdb_path)

# Select Seamount, Seamount Ridges and Canyon
seamounts <- model$Feature[model$Feature %in% c("Seamount", "Seamount Ridges")]
canyons <- model$Feature[model$Feature %in% c("Canyon")]
others <- model$Feature[model$Feature %in% c("Coastal/Shelf Terrane", "Cross Shelf Valley",
                                            "Ridge", "Shelf Deep", "Trough Mouth Fan", 
                                            "Contourite Feature", "Plateau", "")]


In [ ]:
print(model)


In [ ]:
unique(model$Feature)
table(model$Feature)
head(model$Shape_Area,5)
head(model$Shape_Length,5)


In [ ]:
model$Shape

In [ ]:
## HELP
# https://r.geocompx.org/solutions/read-write.html
# https://github.com/highered-esricanada/r-arcgis-tutorials/blob/master/3-R-ArcGIS-Scripting.pdf
# = https://esricanada-ce.github.io/r-arcgis-tutorials/3-R-ArcGIS-Scripting.pdf


In [ ]:
# View
plot(model)


In [ ]:
# List layers inside the geodatabase
gbd.layers <- st_layers(gdb_path)
print(gbd.layers)

In [ ]:
# Analysis
summary(model)

In [ ]:
# Check for the projection
st_crs(model)

In [ ]:
plot(st_geometry(model), border="#aaaaaa", main="Census tracts around city center,\nclipped by 2km buffer ")

st_crs(model)$proj4string

plot(st_geometry(model), pch = 20, col="#ccc", 
     axes=F, main = "before transform - WGS84")
axis(side = 1, las = 3) # adjust text on axes
axis(side = 2, las = 1)


In [ ]:
ext(model)
ncell(model)

In [ ]:
#st_point(c(1750160, 467499.9)) %>% # point coordinates
#  st_sfc(crs = st_crs(model))  # create feature collection, setting CRS to philly_sf's CRS


model_ctr <- st_point(c(1750160, 467499.9)) %>%
    st_sfc(crs = st_crs(model))
st_crs(model_ctr)$proj4string
model_buf <-  st_buffer(model_ctr, 10000)
model_sel <- st_filter(model, model_buf)


In [ ]:
plot(st_geometry(model), border="#aaaaaa", main="Census tracts around city center,\nclipped by 2km buffer ")
plot(st_geometry(model_sel), add=T, col="red")
plot(st_geometry(model_buf), add=T, lwd = 2)


In [ ]:
model_intersection <- st_intersection(model_buf, model)
model_intersection

plot(st_geometry(model), border="#aaaaaa", main="Census tracts around city center,\nclipped by 2km buffer ")
plot(model_intersection, add=T, lwd = 2, border = "red")


In [ ]:
# If it's not WGS84 (EPSG:4326)

forms_to_coords <- function(model_sf){
    # data <- st_transform(model, 4326)
    # data <- st_make_valid(data)
    data <- st_make_valid(model_sf)
    
    # ⚠️ If geometry is NOT points
    #🔸 For polygons or lines
    centroids <- st_centroid(data)
    coords <- st_coordinates(centroids)
    return(coords)
}


In [ ]:
# Export to 
# write.csv(st_drop_geometry(data), paste("outputs/collection/st_drop_geometry.csv"), row.names = FALSE)

coords <- forms_to_coords(model)
head(coords, 5)

In [ ]:
range(coords)

In [ ]:
# If it's not WGS84 (EPSG:4326)

forms_to_coords <- function(model_sf, std.WGS84 = TRUE){
    coords <- st_transform(model_sf, 4326) %>%
        st_make_valid() %>%
        st_centroid() %>%
        st_coordinates()
    colnames(coords) <- c("lon","lat")
    return(coords)
}


In [ ]:
# ==============================================================================
# 1. COMPUTE DISTANCE TO CANYON/ SEAMOUNT
# ==============================================================================

# All points
plot(coords)

# Split canyon vs other features
# Canyon = Cross Shelf Valley
canyons <- model[model$Feature == "Canyon", ]
canyon_coords <- forms_to_coords(canyons)
plot(canyon_coords,add=T, pch = 3, col="black")


# Split seamounts vs other features
seamount_ridges <- model[model$Feature == "Seamount Ridges", ]
seamount_ridges_coords <- forms_to_coords(seamount_ridges)
# plot(seamount_ridges_coords,add=T, col="blue")

# Split seamount ridges vs other features
seamounts <- model[model$Feature == "Seamount", ]
seamounts_coords <- forms_to_coords(seamounts)
# plot(seamounts_coords,add=T, col="green")

# Split seamounts vs other features
seamount_and_Sridges <- model[model$Feature %in% c("Seamount Ridges", "Seamount"), ]
seamount_and_Sridges_coords <- forms_to_coords(seamount_and_Sridges)
points(seamount_and_Sridges_coords, pch = 16, col ="red")



In [ ]:
# ==============================================================================
# ADD FEATURES TO THE GRID
# ==============================================================================

# Convert data (gdb layer) -> SpatVector → sf + Make sure polygons are valid
v_sf <- vect(gdb_path, layer=gbd.layers$name) %>%
    st_as_sf() %>%
    st_transform(4326) %>%
    st_make_valid()

# Ensure CRS is lon/lat + Convert your grid to sf points
grid_sf <- st_as_sf(
  grid,
  coords = c("lon", "lat"),
  crs = 4326
)

# Geometry issue : polygons in GDB geometrically invalid -> Disable s2
sf_use_s2(FALSE)

# Spatial join
grid_joined <- st_join(grid_sf, v_sf["Feature"])

# Back to table
grid_final <- as.data.table(
  cbind(
    st_coordinates(grid_joined),
    st_drop_geometry(grid_joined)
  )
)

# ================
# HANDLE NA VALUES
# ================

# Ensure correct column names
if(!all(c("lon","lat") %in% names(grid_final))){
  setnames(grid_final, c("X","Y"), c("lon","lat"))
}

# Identify NA rows
na_idx <- which(is.na(grid_final$Feature))

# Convert ONLY those points to sf
pts_na_sf <- st_as_sf(
  grid_final[na_idx],
  coords = c("lon","lat"),
  crs = 4326
)

# Find nearest polygon
nearest_idx <- st_nearest_feature(pts_na_sf, v_sf)

## Write back into grid_final
grid_final$Feature[na_idx] <- v_sf$Feature[nearest_idx]


In [ ]:
print("Dim | Without NA; With NA:")
print(dim(na.omit(grid_final))) ; print(dim(grid_final))

which(is.na(grid_final), arr.ind=TRUE)


In [ ]:
# OPTION 1 — Assign nearest feature (most common ✔️)
# Find nearest polygon for NA points
na_idx <- which(is.na(grid_joined$Feature))

nearest_feat <- st_nearest_feature(
  grid_sf[na_idx, ],
  v_sf
)

# Assign feature
grid_joined$Feature[na_idx] <- v_sf$Feature[nearest_feat]

# OPTION 2 — Distance-based assignment (better scientifically 🔥)
# Distance to canyon
canyon_sf <- v_sf[v_sf$Feature == "Canyon", ]

nearest_canyon <- st_nearest_feature(grid_sf, canyon_sf)

grid_joined$dist_canyon <- as.numeric(
  st_distance(
    grid_sf,
    canyon_sf[nearest_canyon, ],
    by_element = TRUE
  )
)

# OPTION 3 — Buffer polygons (spatial expansion)
# v_buffer <- st_buffer(v_sf, dist = 10000)  # 10 km buffer
# grid_joined <- st_join(grid_sf, v_buffer["Feature"])

In [ ]:
head(grid_sf)

In [ ]:
head(canyon_coords) ; plot(canyon_coords)

In [ ]:
# Compute distances
# Other features (or all, depending on your goal)
dist_matrix <- st_distance(coords, canyon_coords)
min_dist <- apply(dist_matrix, 1, min)

In [ ]:
# Extract minimum distance
other$dist_to_canyon <- as.numeric(min_dist)

In [ ]:
head(min_dist)

In [ ]:
forms_to_points <- function(model_sf){
    model_sf %>%
        st_transform(4326) %>%
        st_make_valid() %>%
        st_centroid()
}

In [ ]:
points_all <- forms_to_points(model) %>%
    st_transform(3031)

canyon_pts <- forms_to_points(canyons) %>%
    st_transform(3031)
seamount_pts <- forms_to_points(seamount_and_Sridges) %>%
    st_transform(3031)


In [ ]:
nearest_idx <- st_nearest_feature(points_all, canyon_pts)

min_dist <- st_distance(
  points_all,
  canyon_pts[nearest_idx, ],
  by_element = TRUE
)

points_all$dist_to_canyon <- as.numeric(min_dist)

nearest_idx <- st_nearest_feature(points_all, seamount_pts)

min_dist <- st_distance(
  points_all,
  seamount_pts[nearest_idx, ],
  by_element = TRUE
)

points_all$dist_to_seamount <- as.numeric(min_dist)

In [ ]:
# points_all$dist_to_canyon
# points_all$dist_to_seamount
# points_all$Feature

plot(points_all)

In [ ]:
# ==============================================================================
# DOWNLOAD AND LOAD BATHYMETRY
# ==============================================================================

options(timeout = 600)  # 10 minutes
bathy_file <- "global_topo_1min_topo_19_1.nc"
if(file.exists(bathy_file)){
    cat(bathy_file, "is (are) already in your repertory.")
} else {
    download.file("https://topex.ucsd.edu/pub/global_topo_1min/topo_19.1.nc", bathy_file, mode = "wb")
    print('File Downloaded')
}

# MOVE TO DATA DIRECTORY 
ifelse(!dir.exists(file.path(dataDir)),
        dir.create(file.path(dataDir)),
        "Directory Exists")

file.rename(from=bathy_file,
            to=paste(dataDir, bathy_file, sep = "/"))
bathy_file <- paste(dataDir, bathy_file, sep = "/")

In [ ]:
library(ncdf4)

# Open the NetCDF file
nc_file <- nc_open(bathy_file)

# Get coordinates variables
dim_lon <- ncvar_get(nc_file, "lon", collapse_degen=FALSE)
dim_lat <- ncvar_get(nc_file, "lat", collapse_degen=FALSE)
dim_depth <- ncvar_get(nc_file, "z", collapse_degen=FALSE)
coords <- expand.grid(dim_lon, dim_lat)
depth_matrix <- data.frame(cbind(coords, as.vector(dim_depth)))
names(depth_matrix) <- c("lon", "lat", "depth")

nc_close(nc_file)


In [ ]:

# Filter Out the outerbound values
## 1. Depth
Antarctic_depth_coords <- depth_matrix %>%
  filter(
    lon >= 30, lon <= 150,
    lat >= -70, lat <= -60,
    depth <= 0
  )

In [ ]:
new.grid.lon <- seq(from=30.0, to=150.0, by=0.1)
new.grid.lat <- seq(from=-70.0, to=-60.0, by=0.1)
new.grid <- expand.grid(lon=new.grid.lon, lat=new.grid.lat)

new.grid$depth <- NA
# new.grid$geoFeat <- NA
# new.grid$canyonDist <- NA
# new.grid$seamountDist <- NA


grid <- as.data.table(new.grid)
depth_dt <- as.data.table(Antarctic_depth_coords)

nn <- get.knnx(
  data  = as.matrix(depth_dt[, .(lon, lat)]),
  query = as.matrix(grid[, .(lon, lat)]),
  k = 1
)
# Assign interpolated values
grid[, depth := depth_dt$depth[nn$nn.index]]

In [ ]:
head(grid)

In [ ]:
grid_pts <- vect(grid, geom = c("lon", "lat"), crs = "EPSG:4326")
res <- terra::extract(v_ll, grid_pts)
grid$Feature <- res$Feature

head(grid)

# Statistical Analyses

In [ ]:
system("conda install conda-forge::r-cluster conda-forge::r-remotes conda-forge::r-devtools")


In [ ]:
install.packages("remotes")
remotes::install_github("bioRgeo/bioregion")


In [ ]:
library(cluster)
library(bioregion)

In [ ]:
# Default value of dissimilarity matrix

# -----------------------
# clean SST flag
# underwater.grid[sst == -32767, sst := NA]
# -----------------------

rdm.rows <- sample(nrow(underwater.grid), 900, replace = FALSE)
X <- underwater.grid[rdm.rows, .(sst, depth, iceC)]

# compute Gower distances (small subset!)
distance.matrix <- cluster::daisy(X, metric = "gower")
gower_matrix <- as.matrix(distance.matrix)

subset_clust <- 200
clara.object <- bioregion::nhclu_clara(
    dissimilarity = distance.matrix, n_clust = subset_clust)
                                       


In [ ]:
clusters900 <- underwater.grid[rdm.rows, ]
clusters900$K_200 <- clara.object$clusters$subset_clust

In [ ]:
summarise.n.cluster <- function(df, cluster.column){
    summarised.df <- df %>%
        # Here, we assume that EVERY k-cluster starts with "K_"
        mutate(across(starts_with("K_"), as.integer)) %>%
        group_by({{cluster.column}}) %>%

        # Create K-Centroids
        # Here, we assume that df contains sst, depth, iceC
        summarise(
            mean_sst   = mean(sst, na.rm = TRUE),
            mean_depth = mean(depth, na.rm = TRUE),
            mean_iceC  = mean(iceC, na.rm = TRUE)
        )
    return(summarised.df)
}

upgma.classification <- function(df){
    data.table.var <- as.data.table(df)
    distance.matrix <- cluster::daisy(data.table.var, metric = "gower")
    hclust.K <- hclust(d = distance.matrix, method = "average")

    return(hclust.K)
    }

upgma.display.branch <- function(df, nb_clust = 9){
    hc <- upgma.classification(df)
    dend <- as.dendrogram(hc)
    upgma.cluster <- cutree(hc, k = nb_clust)
    cols <- rainbow(length(unique(upgma.cluster)))
    dend <- color_branches(dend, k = nb_clust, col = cols)
    plot(dend, cex = 0.6, main = "UPGMA Tree - Branches colored by cluster")
}

upgma.display.rect <- function(df, nb_clust = 9){
    hc <- upgma.classification()
    plot(hc, hang = -1, labels = FALSE,ncex = 0.6, main = paste("UPGMA Tree with",cutoff,"clusters"))
    rect.hclust(hc, k = nb_clust, border = rainbow(9), lw = 2)
    text(x = -0.05, y = 0.5, labels = 1:8, col = rainbow(8), lwd = 2, cex = 1.2)
}

infer.clust <- function(df = clusters900, custer.col = K_200, hclust.benthic = hclust.benthic.K200, cutoff = 9, x_name = "K_200"){
    X.1 <- summarise.n.cluster(df = df, cluster.column = {{custer.col}}) %>%
        rename(cluster_id = {{custer.col}})
    hclust.benthic$labels <- X.1$cluster_id
    
    groups <- cutree(hclust.benthic, k = cutoff)
    
    group_map <- data.frame(
      cluster_id = as.character(names(groups)),
      group_id = groups
    )
    
    df_with_groups <- merge(
      df,
      group_map,
      by.x = x_name,
      by.y = "cluster_id"
    )
    return(df_with_groups)
}



In [ ]:
df_K200 <- summarise.n.cluster(df = clusters900, cluster.column = K_200)
hclust.benthic.K200 <- upgma.classification(df_K200)

cutoff <- 12
plot(hclust.benthic.K200, hang = -1, cex = 0.6, main = paste("UPGMA Tree with",cutoff,"clusters"))
rect.hclust(hclust.benthic.K200, k = cutoff, border = seq(from = 1, to = 9, by = 1))

In [ ]:
clusters900_with_groups <- infer.clust(
    df = clusters900, custer.col = K_200,
    hclust.benthic = hclust.benthic.K200,
    cutoff = 9, x_name = "K_200")

k.centroids <- clusters900_with_groups %>%
  group_by(group_id) %>%
    summarise(
        mean_canDist   = mean(canyonDist, na.rm = TRUE),
        mean_depth = mean(depth, na.rm = TRUE),
        mean_seamtDist  = mean(seamountDist, na.rm = TRUE)
    )

# Centroids
centroids <- k.centroids[, c("mean_canDist", "mean_depth", "mean_seamtDist")]
colnames(centroids) <- c("canyonDist", "depth", "seamountDist")
# Grid data
grid_vals <- underwater.grid[, c("canyonDist", "depth", "seamountDist")]

# ------------------------------------------------------------------------------
# Combine for consistent scaling
scaled <- rbind(grid_vals, centroids) %>%
    scale()

grid_scaled <- scaled[1:nrow(grid_vals), ]
centroids_scaled <- scaled[(nrow(grid_vals)+1):nrow(scaled), ]

# Then run nearest neighbor
nn <- get.knnx(data = centroids_scaled, query = grid_scaled, k = 1)
underwater.grid$cluster <- nn$nn.index[,1]

export.csv(directory = outputDir, gower_matrix, "BenthicData_gower_K50.txt")

# Export GeoTIFF

In [ ]:
system("conda install -y conda-forge::r-hrbrthemes conda-forge::r-viridis")


In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)
library(viridis)
library(hrbrthemes)


In [ ]:
# Convert to data.frame (terra prefers that)
grid_df <- as.data.frame(underwater.grid)

# Create raster from XYZ (lon, lat, value)
r_canyon   <- rast(grid_df[, c("lon", "lat", "canyonDist")], type = "xyz")
r_depth <- rast(grid_df[, c("lon", "lat", "depth")], type = "xyz")
r_seamount  <- rast(grid_df[, c("lon", "lat", "seamountDist")], type = "xyz")
r_cluster <- rast(grid_df[, c("lon", "lat", "cluster")], type = "xyz")

r_stack <- c(r_canyon, r_depth, r_seamount, r_cluster)
for(var in r_stack){
    crs(var)   <- "EPSG:4326"
}

names(r_stack) <- c("distCanyon", "depth", "distSeamount", "cluster")

In [ ]:
# ---- 1. Prepare data ----
grid_long <- grid_df %>%
  pivot_longer(
    cols = c(depth, sst, iceC),  # rename if needed
    names_to = "variable",
    values_to = "value"
  ) %>%
  mutate(
    cluster = factor(cluster, levels = 1:12),   # adapt if not 12
    variable = recode(variable,
                      depth = "Depth (m)",
                      sst = "Sea surface temperature (°C)",
                      iceC = "Sea Ice")
  )

# ---- 2. Palette (consistent across everything) ----
pal <- setNames(hcl.colors(length(levels(grid_long$cluster)), "Set3"),
                levels(grid_long$cluster))

# ---- 3. Plot ----
p <- ggplot(grid_long,
            aes(x = value, y = cluster, fill = cluster)) +
  
  geom_violin(scale = "width", width = 0.8, color = "black", size = 0.2) +
  facet_wrap(~ variable, nrow = 1, scales = "free_x") +
  scale_fill_manual(values = pal) +
  labs(x = NULL, y = "Cluster number") +
  theme_minimal(base_size = 12) +
  theme(
    legend.position = "none",
    strip.text = element_text(face = "bold"),
    panel.spacing = unit(1.5, "lines")
  )

p

# ---- 4. Export ----
ggsave("violin_panels.png", p, width = 12, height = 6, dpi = 300)

In [ ]:
export.geomap <- function(output.directory = outputDir, filename, column.name, plot.title){
    png.name <- paste(output.directory,"/",filename,".png", sep="")
    png(png.name, width = 800, height = 600)
    plot(r_stack[[column.name]], main = plot.title)
    dev.off()
}

In [ ]:
# 1
png("outputs/collection/depth.png", width = 800, height = 600)
plot(r_stack[["depth"]], main = "Bathymetry")
dev.off()

# 2
png("outputs/collection/distSeamount.png", width = 800, height = 600)
# plot(r_stack[["sst"]], main = "Sea Surface Temperature")
dev.off()

# 3
png("outputs/collection/distCanyon.png", width = 800, height = 600)
# plot(r_stack[["iceC"]], main = "Ice Concentration")
dev.off()

# Benthic bioregions
png("outputs/collection/BenBioregion.png", width = 800, height = 600)
cols <- hcl.colors(12, "Set3")  # best default choice
plot(  r_stack[["cluster"]],  col = cols,
  breaks = seq(0.5, 12.5, by = 1),  # important for discrete classes
  main = "Benthic bioregions"
)
dev.off()